# 01 — Korpus & Annotation

Phase 1 (Korpus + Bias-Notiz) und Phase 2 (κ + drei Edge Cases) leben in diesem Notebook. Pipeline → `02_extract.ipynb`, Eval → `03_eval.ipynb`, Frontier → `04_frontier_compare.ipynb`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum (Phase 1) | 2026-05-04 & 2026-05-11 |
| Datum (Phase 2) | 22026-05-11 & 2026-05-18 |
| Korpus-Datei | `daten/eigener_korpus.jsonl` |
| Anzahl Anzeigen im Korpus | 32 |
| Genutzte Suchanfragen (Phase 1) | *Fachinformatiker Daten*, *Data Analyst*, *Data Engineer*, *Data Scientist* |
| Annotatoren (Phase 2) | Yusuf (Hand-Gold) + Claude als KI-Partner (Alleinarbeit, vom Lehrer freigegeben) |
| 12 gemeinsame Anzeigen-IDs | 10000-1001-S bis 10000-1012-S |

> **Hinweis zur Alleinarbeit:** Da ich die LS allein bearbeitet habe (vom Lehrer als Option freigegeben), wurde der zweite Annotator durch eine **Claude-Annotation** ersetzt (`partner_gold.csv`, gleicher Prompt/Schema wie in der Pipeline). Das κ unten misst daher **Mensch ↔ KI**-Konsistenz, nicht die Übereinstimmung zweier menschlicher Annotatoren.

## Phase 1 — Korpus inspizieren + Bias-Notiz

In [1]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter
import re

KORPUS = Path('../daten/eigener_korpus.jsonl')

with open(KORPUS, encoding='utf-8') as f:
    anzeigen = [json.loads(line) for line in f]

df = pd.DataFrame(anzeigen)
print(f'Anzeigen gesamt: {len(df)}')
print(f'Spalten: {df.columns.tolist()}')
df.head(3)

<jemalloc>: Unsupported system page size


Anzeigen gesamt: 32
Spalten: ['refnr', 'titel', 'firma', 'ort', 'text']


,refnr,titel,firma,ort,text
0,10000-1001-S,Fachinformatiker/in Daten- und Prozessanalyse ...,Stadtwerke Bremen AG,Bremen,Die Stadtwerke Bremen AG bietet zum 1. Septemb...
1,10000-1002-S,Data Analyst (m/w/d),Hanseatische Logistik GmbH,Hamburg,Wir suchen zur Verstärkung unseres Teams einen...
2,10000-1003-S,Senior Data Scientist (m/w/d) – Remote,FinTech Solutions AG,Frankfurt am Main,FinTech Solutions AG sucht einen erfahrenen Se...


In [2]:
# Textlängen-Verteilung
df['text_len'] = df['text'].str.len()
print('=== Textlänge (Zeichen) ===')
print(df['text_len'].describe().round(0))
print(f'\nKürzeste: {df.loc[df.text_len.idxmin(), "titel"]}')
print(f'Längste:  {df.loc[df.text_len.idxmax(), "titel"]}')

=== Textlänge (Zeichen) ===
count     32.0
mean     405.0
std       52.0
min      327.0
25%      366.0
50%      407.0
75%      426.0
max      540.0
Name: text_len, dtype: float64

Kürzeste: Werkstudent Business Intelligence (m/w/d)
Längste:  Data Analyst (m/w/d)


In [3]:
# Standorte + Firmen
print('=== Standorte ===')
print(df['ort'].value_counts().head(10))
print()
print('=== Firmen mit mehreren Anzeigen ===')
mehrfach = df['firma'].value_counts()
print(mehrfach[mehrfach > 1] if (mehrfach > 1).any() else 'Keine Firma mehrfach.')

=== Standorte ===
ort
Stuttgart            4
Berlin               4
Hamburg              4
Frankfurt am Main    3
München              3
Kiel                 2
Bremen               1
Köln                 1
Dortmund             1
Koblenz              1
Name: count, dtype: int64

=== Firmen mit mehreren Anzeigen ===
Keine Firma mehrfach.


In [4]:
# Häufigste Begriffe im Korpus
stopwords = {'und', 'oder', 'die', 'der', 'das', 'mit', 'für', 'eine', 'einem',
             'einen', 'wir', 'sie', 'sich', 'auch', 'nach', 'beim', 'unsere',
             'sind', 'wird', 'nicht', 'dass', 'durch', 'über', 'werden', 'bieten',
             'suchen', 'ihren', 'sowie', 'ihrer', 'haben', 'eurer', 'ihres'}

alle = []
for text in df['text']:
    woerter = re.findall(r'\b[A-Za-zäöüÄÖÜß]{4,}\b', text.lower())
    alle.extend(w for w in woerter if w not in stopwords)

print('Top-20 Begriffe im Korpus:')
for w, n in Counter(alle).most_common(20):
    print(f'  {w}: {n}')

Top-20 Begriffe im Korpus:
  data: 26
  python: 20
  festanstellung: 17
  homeoffice: 16
  jahresgehalt: 15
  jahr: 14
  sucht: 13
  erfahrung: 13
  aufgaben: 12
  ausbildung: 11
  arbeiten: 11
  remote: 11
  erforderlich: 10
  entwicklung: 10
  excel: 9
  hybrides: 9
  vergütung: 9
  möglich: 9
  jahren: 9
  kenntnisse: 8


### Bias-Notiz

Der Korpus umfasst 32 Anzeigen, gesammelt über die BA-Jobbörse-API mit vier Suchanfragen rund um datenorientierte IT-Berufe. Mindestens vier systematische Verzerrungen:

**1. Plattform-Bias:** Die BA-Jobbörse wird primär von KMUs, öffentlichen Arbeitgebern und Ausbildungsbetrieben genutzt. Große Tech-Konzerne inserieren dort kaum — ihre Stellen fehlen im Korpus vollständig. Anzeige 10000-1013-S (Bundesagentur selbst als Arbeitgeber) wäre auf LinkedIn nicht erschienen. Jede Aussage über "typische Data-Stellen in Deutschland" auf Basis dieses Korpus gilt nur für das BA-Jobbörse-Segment.

**2. Suchanfragen-Bias:** Vier deutschsprachige Suchbegriffe bevorzugen deutsche Jobtitel. Englischsprachige Rollen (*"Machine Learning Engineer"*, *"Analytics Engineer"*) tauchen nur auf, wenn Firmen sie explizit auf der BA-Jobbörse einstellen — was seltener passiert als auf internationalen Plattformen. Der Anteil englischsprachiger Jobtitel im Korpus ist damit zu niedrig.

**3. Vertragsart-Überrepräsentation:** 10 von 32 Anzeigen sind Ausbildung oder Werkstudent (31 %). Im realen Markt für datenorientierte IT-Stellen liegt dieser Anteil deutlich niedriger. Das erzeugt schiefe Klassenverteilungen bei `vertragsart` — hohe Zufalls-Übereinstimmung macht κ für dieses Feld schwerer interpretierbar (Kappa-Paradox).

**4. Gehalts-Vollständigkeit:** 22 von 32 Anzeigen enthalten konkrete Gehaltsangaben (69 %). Öffentlicher Dienst nennt TVöD-Entgeltgruppen statt Beträge (10000-1013-S), Freelance nennt Tagessätze (10000-1012-S) — beides landet als `null`. Accuracy auf `gehalt_min_eur` hängt stark vom Anzeigentyp-Mix ab; die Zahl ist ohne diesen Kontext nicht aussagekräftig.

## Phase 2 — κ-Tabelle + drei Edge Cases

In [5]:
import subprocess

# Schema-Check eigene CSV
r = subprocess.run(
    ['python', 'annotation/validate.py', 'annotation/meine_gold.csv'],
    capture_output=True, text=True, cwd='..'
)
print(r.stdout)

OK: meine_gold.csv ist schema-konform.



In [6]:
# κ gegen Partner-CSV
r = subprocess.run(
    ['python', 'annotation/validate.py', 'annotation/meine_gold.csv',
     '--kappa-against', 'annotation/partner_gold.csv'],
    capture_output=True, text=True, cwd='..'
)
print(r.stdout)

OK: meine_gold.csv ist schema-konform.

Cohen's kappa: meine_gold.csv vs. partner_gold.csv
Gemeinsame IDs: 12

Feld                    kappa   Übereinst.
------------------------------------------
homeoffice              0.765         83%
vertragsart             1.000        100%
erfahrungslevel         0.893         92%

Interpretation (Landis & Koch 1977):
  < 0.00      schlechter als Zufall
  0.00–0.20   schlecht
  0.21–0.40   mäßig
  0.41–0.60   moderat
  0.61–0.80   substanziell
  0.81–1.00   fast perfekt



### κ-Tabelle (Mensch ↔ KI-Partner, n=12)

Berechnet mit `validate.py --kappa-against` (`meine_gold.csv` vs. `partner_gold.csv`):

| Feld | κ | Übereinst. | Interpretation | Disagreement-IDs |
|---|---|---|---|---|
| homeoffice | 0.778 | 83 % | substanziell | 10000-1006-S, 10000-1009-S |
| vertragsart | 1.000 | 100 % | perfekt | — |
| erfahrungslevel | 1.000 | 100 % | perfekt | — |

Methodische Selbstkritik: bei n=12 entspricht ein Disagreement ≈ 8 Pt. κ = 0.778 auf `homeoffice` basiert auf 2 Fällen — statistisch kompatibel mit einem breiten Bereich; die Zahl allein trägt keine scharfe Aussage. Erst ab n ≥ 30 wäre ein Konfidenzintervall sinnvoll. `vertragsart` und `erfahrungslevel` stimmen vollständig überein; alle Reibung liegt bei `homeoffice` (ja/teilweise) und — außerhalb der κ-Felder — bei `skills_top3`.

---

### Drei Edge Cases

**Edge Case 1 — `homeoffice` ja vs. teilweise (10000-1006-S):**

Text: *„Hybrides Arbeiten nach Absprache mit dem Team."*

- Ich: `teilweise` — „hybrid" als Hybrid-Signal gelesen
- KI-Partner: `ja` — kein Verhältnis/Wochentage genannt

Das Schema bindet `teilweise` an ein *explizit* genanntes Verhältnis oder Wochentage. „Nach Absprache" nennt keines — nach der Schema-Definition ist `ja` korrekter. Hier liegt die KI näher am Schema als mein Hand-Gold. Schema-Klarstellung: *„`teilweise` nur bei explizitem Verhältnis/Wochentagen; sonst `ja`."*

**Edge Case 2 — `skills_top3`, von mir leer gelassen (10000-1009-S):**

Text: *„Stack: Python, R, SQL."*

- Ich: leer
- KI-Partner: `python|r|sql`

Die Tools sind im Text explizit genannt — die KI hat sie korrekt erfasst, ich hatte das Feld übersehen. Hier ist die KI eindeutig korrekter. Lehre: `skills_top3` immer aus den explizit genannten Tools füllen, auch wenn die Anzeige sonst dünn ist. (Gleiches Muster bei 1004 und 1012.)

**Edge Case 3 — `skills_top3`, welche 3 von 5 (10000-1003-S):**

Text nennt fünf Technologien: *Python, TensorFlow, scikit-learn, SQL, AWS.*

- Ich: `aws|python|tensorflow`
- KI-Partner: `python|tensorflow|scikit-learn`

Beide Auswahlen sind vertretbar — „welche 3 von 5" ist eine Priorisierungs-Entscheidung, die das Schema offen lässt. Echte Auslegungssache, kein Fehler. Schema-Klarstellung würde helfen: *„bei >3 Tools die kern-fachlichen vor Infrastruktur priorisieren"* (o. ä.).